# Advanced technical component — Visual Image Retrieval (Project 9)

**Team:** KAUST Academy AI251P — Project 9 (five-member team)  
**Extension type:** Embedding-based visual similarity retrieval (not the standard classification task).  
**Question:** Given a waste image from the held-out TrashNet test split, can we retrieve **visually similar waste materials** from the training split?

We use a **frozen ImageNet-pretrained ResNet18** to extract a 512-dimensional descriptor per image, L2-normalise the descriptors, and rank the training gallery using cosine similarity. We compute **Precision@1, Precision@5, Hit@5, mAP@10, and MRR@10**, plus per-class results, a random-gallery expected baseline, and sample retrieved-neighbour galleries.

**Data integrity:** the embedded CSV is the previously recorded group-safe `split_manifest.csv` from this team's real Phase 1 runs (2,524 images, seed 42). Train images are gallery items; test images are queries. Test labels are only used for evaluation — not for tuning model parameters, choosing checkpoints or ranking neighbours. No dataset images or checkpoint weights are redistributed in this notebook.

**IMPORTANT — evidence status:** This notebook contains working experiment code, **not precomputed retrieval results**. Run all cells on actual images, then download and review the generated metrics before claiming this advanced component as completed. Five-member credit and suitability remain subject to instructor approval.

Suggested setup: Google Colab → Runtime → Change runtime type → T4 GPU if available → Run all.

In [ ]:
%pip -q install "huggingface_hub>=0.26,<2" "pandas>=2.0" "scikit-learn>=1.4" "tqdm>=4.66" "matplotlib>=3.7"

import os, sys, io, json, time, base64, gzip, hashlib, random, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch, torchvision
from torch import nn
from torchvision import transforms, models
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/content/KAUST_Retrieval_Extension') if Path('/content').exists() else Path.cwd()/'KAUST_Retrieval_Extension'
OUT.mkdir(parents=True,exist_ok=True)
print('Device:',DEVICE,'Torch:',torch.__version__,'Output folder:',OUT)

## Recorded dataset split — embedded so the experiment does not need Google Drive or a private repository

In [ ]:
MANIFEST_B64 = (
    '...'
)
manifest_bytes=gzip.decompress(base64.b64decode(MANIFEST_B64))
(OUT/'split_manifest.csv').write_bytes(manifest_bytes)
manifest=pd.read_csv(io.BytesIO(manifest_bytes))
assert len(manifest)==2524, f'Expected 2524 post-cleaning images, got {len(manifest)}'
assert set(manifest['split'])=={'train','val','test'}
assert manifest.groupby('group_id')['split'].nunique().max()==1, 'Group leakage in recorded split!'
print('Recorded split counts:',manifest['split'].value_counts().to_dict())
print('Classes:',sorted(manifest['label_name'].unique().tolist()))

## Download original TrashNet and verify exact image identity

In [ ]:
# Obtain TrashNet from its public upstream source. Never commit or redistribute raw photos.
DATA_ARCHIVE=Path('/content/dataset-resized.zip')
if not DATA_ARCHIVE.exists():
    try:
        DATA_ARCHIVE=Path(hf_hub_download(repo_id='garythung/trashnet',repo_type='dataset',filename='dataset-resized.zip'))
    except Exception as e:
        from urllib.request import urlretrieve
        print('Hugging Face unavailable; trying upstream GitHub:',type(e).__name__)
        DATA_ARCHIVE=Path('/content/dataset-resized.zip')
        urlretrieve('https://github.com/garythung/trashnet/raw/refs/heads/master/data/dataset-resized.zip', DATA_ARCHIVE)
if not zipfile.is_zipfile(DATA_ARCHIVE):
    raise RuntimeError('TrashNet download failed. Upload dataset-resized.zip to /content and re-run this cell.')
DATA_ROOT=Path('/content/KAUST_TrashNet_Extracted')
DATA_ROOT.mkdir(exist_ok=True)
if not list(DATA_ROOT.rglob('*.jpg')):
    with zipfile.ZipFile(DATA_ARCHIVE) as source_zip:
        for entry in source_zip.infolist():
            target=(DATA_ROOT/entry.filename).resolve()
            if not str(target).startswith(str(DATA_ROOT.resolve())+os.sep):
                raise ValueError('Unsafe zip archive path')
        source_zip.extractall(DATA_ROOT)

classes=['cardboard','glass','metal','paper','plastic','trash']
all_images=[p for p in DATA_ROOT.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png') and p.parent.name.lower() in classes]
image_map={(p.parent.name.lower(),p.name):p for p in all_images}
if len(image_map)!=len(all_images):
    raise ValueError('Ambiguous image filenames. Resolve before continuing.')
manifest['resolved_path']=[str(image_map.get((str(r.label_name),Path(r.path).name),'')) for r in manifest.itertuples()]
missing=manifest[manifest.resolved_path=='']
if len(missing):
    raise FileNotFoundError(f'{len(missing)} original images absent; first: {missing.iloc[0].path}')
assert manifest.groupby('split').size().to_dict()=={'test':507,'train':1509,'val':508}
# Verifies that the locally downloaded images match recorded provenance.
for row in tqdm(manifest.itertuples(),total=len(manifest),desc='Verify original hashes'):
    with open(row.resolved_path,'rb') as f:
        observed=hashlib.file_digest(f,'sha256').hexdigest()
    if observed!=row.sha256:
        raise ValueError(f'File SHA256 differs from original experiment manifest: {row.resolved_path}')
print('All',len(manifest),'images match original recorded SHA256 hashes.')

## Extract 512-dimensional embeddings (fixed pretrained model, optionally original fine-tune checkpoint)

In [ ]:
# OPTIONAL: if you already have best_model.pt on the Colab runtime at /content/best_model.pt,
# the code uses the original fine-tuned ResNet18. Otherwise it uses a separate fixed
# ImageNet-pretrained ResNet18 baseline. No images, test labels, or secrets are uploaded.
CKPT=Path('/content/best_model.pt')
FEATURE_SOURCE='fine_tuned_ResNet18' if CKPT.is_file() else 'ImageNet_ResNet18'
if CKPT.is_file():
    checkpoint=torch.load(CKPT,map_location='cpu',weights_only=True)
    assert checkpoint.get('kind')=='finetune'
    assert checkpoint.get('classes')==classes
    model=models.resnet18(weights=None)
    model.fc=nn.Linear(model.fc.in_features,len(classes))
    model.load_state_dict(checkpoint['state_dict'],strict=True)
    model.fc=nn.Identity()
    IMG_SIZE=int(checkpoint.get('img_size',160))
else:
    # Pre-trained ImageNet weights load automatically on Colab.
    model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc=nn.Identity()
    IMG_SIZE=160
model=model.to(DEVICE).eval()
for p in model.parameters():p.requires_grad_(False)
transform=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225]),
])

class WasteImageDS(Dataset):
    def __init__(self,part):self.frame=part.reset_index(drop=True)
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        item=self.frame.iloc[i]
        with Image.open(item.resolved_path) as img:
            x=transform(img.convert('RGB'))
        return x,int(item.label_id)

@torch.inference_mode()
def embed_split(part):
    dl=DataLoader(WasteImageDS(part),batch_size=64 if DEVICE.type=='cuda' else 16,
                  shuffle=False,num_workers=2,pin_memory=(DEVICE.type=='cuda'))
    vectors=[]; ys=[]
    for image_batch,label_batch in tqdm(dl,desc='Embedding '+part.iloc[0]['split']):
        x=model(image_batch.to(DEVICE))  # 512-d post-average-pool ResNet18 embedding
        x=torch.nn.functional.normalize(x,p=2,dim=1)
        vectors.append(x.cpu().numpy())
        ys.extend(label_batch.tolist())
    return np.concatenate(vectors,axis=0).astype('float32'),np.asarray(ys,dtype=np.int64)

train=manifest[manifest.split=='train'].copy().reset_index(drop=True)
test=manifest[manifest.split=='test'].copy().reset_index(drop=True)
gallery_embeddings,gallery_y=embed_split(train)
query_embeddings,query_y=embed_split(test)
assert gallery_embeddings.shape==(1509,512)
assert query_embeddings.shape==(507,512)
np.savez_compressed(OUT/'retrieval_feature_summary.npz',gallery_shape=gallery_embeddings.shape,
                   query_shape=query_embeddings.shape) # NO images or embeddings published
print('Extracted:',FEATURE_SOURCE,'gallery:',gallery_embeddings.shape,'queries:',query_embeddings.shape)

## Cosine nearest-neighbour retrieval + evaluation on held-out test queries

In [ ]:
# Cosine = dot product because both image descriptors are unit-normalised.
SIM = query_embeddings @ gallery_embeddings.T
assert SIM.shape==(507,1509)
K=10
# Stable sort: descending similarity. No query image is in the gallery (train vs test).
indices=np.argsort(-SIM,axis=1,kind='stable')[:,:K]
neighbour_y=gallery_y[indices]
relevant=(neighbour_y==query_y[:,None])
n_pos=np.bincount(gallery_y,minlength=len(classes))[query_y]
assert np.all(n_pos>0)

precision1=relevant[:,0].astype(float)
precision5=relevant[:,:5].mean(axis=1)
hit5=relevant[:,:5].any(axis=1).astype(float)
# AP@10 is conventional truncated average precision: divide by min(total relevant in gallery, 10).
rank=np.arange(1,K+1)
prec_at_rank=np.cumsum(relevant,axis=1)/rank[None,:]
ap10=(prec_at_rank*relevant).sum(axis=1)/np.minimum(n_pos,K)
rr10=np.where(relevant.any(axis=1),1/(relevant.argmax(axis=1)+1),0.0)

rows=pd.DataFrame({'query_file':[Path(p).name for p in test.resolved_path],
                   'true_label':test.label_name.astype(str),
                   'precision_at_1':precision1,'precision_at_5':precision5,
                   'hit_at_5':hit5,'ap_at_10':ap10,'rr_at_10':rr10,
                   'top_1_label':[classes[c] for c in neighbour_y[:,0]],
                   'top_1_similarity':SIM[np.arange(len(SIM)),indices[:,0]]})
rows.to_csv(OUT/'retrieval_queries.csv',index=False)
metric_columns=['precision_at_1','precision_at_5','hit_at_5','ap_at_10','rr_at_10']
overall=rows[metric_columns].mean().to_dict()
by_class=rows.groupby('true_label')[metric_columns].agg(['mean','count'])
by_class.to_csv(OUT/'retrieval_per_class.csv')
# Random-gallery expectation, computed analytically from fixed class counts.
# Chance of at least one relevant item in five without replacement.
from math import comb
p_random1=np.mean(n_pos/len(gallery_y))
p_randomhit5=np.mean([1 - (comb(len(gallery_y)-int(pos),5)/comb(len(gallery_y),5)) for pos in n_pos])

# Query-level bootstrap confidence intervals; descriptive only, not source-independent inference.
rng=np.random.default_rng(SEED)
bootstrap_idx=rng.integers(0,len(rows),size=(1000,len(rows)))
boots={}
for col in metric_columns:
    estimates=rows[col].to_numpy()[bootstrap_idx].mean(axis=1)
    boots[col]={'low':float(np.quantile(estimates,.025)), 'high':float(np.quantile(estimates,.975))}
summary={'advanced_component':'Embedding-based waste image retrieval',
         'checkpoint_source':FEATURE_SOURCE,'input_size':[IMG_SIZE,IMG_SIZE],
         'data':'TrashNet; original leak-safe recorded split',
         'gallery_split':'train','gallery_size':len(train),'query_split':'test','query_size':len(test),
         'embedding_dimensions':int(gallery_embeddings.shape[1]),
         'similarity':'L2-normalized cosine similarity; no score threshold',
         'metrics':{k:float(v) for k,v in overall.items()},
         'bootstrap_query_ci95':boots,
         'expected_random_p_at_1':float(p_random1),'expected_random_hit_at_5':float(p_randomhit5),
         'seed':SEED,'limitations':'Same-dataset image retrieval; not cross-domain conveyor validation; instructor approval for fifth-member work is separate.'}
(OUT/'retrieval_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))
display(rows.groupby('true_label')[metric_columns].mean().round(3))

## Example retrieval gallery (six deterministic queries — one per class)

In [ ]:
# Inspect a fixed, label-balanced sample of query results — true and nearest labels are shown.
fig,axes=plt.subplots(6,6,figsize=(16,17))
for i,cl in enumerate(classes):
    matches=np.flatnonzero(test.label_name.to_numpy()==cl)
    q=int(matches[0])  # fixed choice, not cherry-picked success
    with Image.open(test.loc[q,'resolved_path']) as img:
        axes[i,0].imshow(img.convert('RGB'))
    axes[i,0].set_title('QUERY (test)\n'+cl,fontsize=9)
    for rank_j in range(5):
        item=int(indices[q,rank_j])
        g=train.iloc[item]
        with Image.open(g.resolved_path) as img:
            axes[i,rank_j+1].imshow(img.convert('RGB'))
        axes[i,rank_j+1].set_title(f'TOP {rank_j+1} (train)\n{g.label_name}  cos {SIM[q,item]:.2f}',fontsize=8)
for ax in axes.flat:ax.axis('off')
fig.suptitle('TrashNet visual retrieval: first test query of each class; five nearest training-gallery images',fontsize=12)
fig.tight_layout(rect=[0,0,1,.97])
fig.savefig(OUT/'retrieval_gallery.png',dpi=135)
plt.show()

## Reproducible results bundle and report insert

In [ ]:
# Create a transparent, paste-ready methods-and-results paragraph using the ACTUAL computed numbers.
metrics=summary['metrics']; src=summary['checkpoint_source']
report_text=(
    'Advanced technical extension — embedding-based visual image retrieval. '
    f'We used {src} as a fixed feature encoder for 512-dimensional, L2-normalized image embeddings at {IMG_SIZE}×{IMG_SIZE} input resolution. '
    'The leakage-aware original TrashNet split was retained, with 1,509 train images as the reference gallery and '
    '507 held-out test images as queries. We ranked the gallery by cosine similarity without a similarity threshold. '
    f'The results were Precision@1={metrics["precision_at_1"]:.3f}, Precision@5={metrics["precision_at_5"]:.3f}, '
    f'Hit@5={metrics["hit_at_5"]:.3f}, mAP@10={metrics["ap_at_10"]:.3f}, and MRR@10={metrics["rr_at_10"]:.3f}. '
    f'The expected random-gallery Precision@1 baseline was {summary["expected_random_p_at_1"]:.3f}. '
    'These values describe same-dataset retrieval rather than performance on ZeroWaste conveyor images; '
    'no additional model training was performed for this retrieval extension.'
)
(OUT/'report_addendum_from_actual_run.txt').write_text(report_text,encoding='utf-8')
print('\n',report_text)

# Download reproducible audit results. Generated metrics describe the real run, not simulated data.
archive=Path('/content/KAUST_Retrieval_Actual_Results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for filename in ['split_manifest.csv','retrieval_queries.csv','retrieval_per_class.csv',
                     'retrieval_metrics.json','retrieval_gallery.png','report_addendum_from_actual_run.txt']:
        z.write(OUT/filename,arcname=filename)
print('Saved results:',archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Not in Colab: download',archive,'manually.')

### How to submit this extension honestly

1. Run every cell on the original TrashNet images. This notebook will download a **results ZIP** containing numerical metrics and a retrieval gallery.
2. Open `retrieval_metrics.json` and `retrieval_gallery.png` to verify that the results correspond to a real run. **Do not report numerical values until the notebook has run.**
3. Add the methods/results paragraph from `report_addendum_from_actual_run.txt` and the retrieval figure to the Overleaf report. State: gallery=train (N=1509), queries=test (N=507), RGB resolution=160×160, ranking by cosine similarity, no score threshold.
4. Upload the `.ipynb` and the generated experiment CSV/JSON to the public **clean** GitHub repository, without raw images, checkpoints, API keys or student IDs. For five-member assessment, obtain instructor acceptance of this extension; implementation alone does not prove fulfillment.

If you want the **fine-tuned classifier embeddings** instead of the fixed ImageNet-pretrained baseline, upload your original `best_model.pt` privately to Colab under `/content/best_model.pt` **before running the model cell**, then rerun from that cell. Never upload checkpoint unless its distribution is permitted.